# Процессы в Python

В Python процессы применяют для распараллеливания тяжёлых вычислительных операций — вычисление хэшей, работа с матрицами, обработка изображений и подобными операциями.

Необходимо отделить точку входа в программу. При создании процесса запускается Python-интерпретатор. Когда подгрузятся модули, он будет выполнять код, который находится на их уровне. То есть, если убрать `if __name__ == '__main__':`, процессы начнут создаваться рекурсивно,  и в итоге они создадут fork-бомбу. К счастью, Python не позволит запустить программу без явной точки входа, и не даст уйти процессу в рекурсию. 

In [11]:
# Код из demo/mp_demo_1.py
import time
import os

import multiprocessing as mp
from multiprocessing import Process

# Функция, которая будет работать в отдельном процессе
def printer(name):
    time.sleep(10)
    print('Привет', name)

if __name__ == '__main__':
    # Запуск в Jupyter Notebook
    mp.set_start_method("fork", force=True)
    # Создаём объект процесса, передав в него функцию и аргументы, с которыми он будет вызван
    p = Process(target=printer, args=('Алиса', ))
    # Запускаем
    p.start()
    print('Пока выполняется процесс, съешьте ещё этих мягких французских булок да выпейте чаю ☕️ ')
    # Выведем Process ID для текущего процесса и для процесса, который только что запустили
    print('Главный PID', os.getpid())
    print('Дочерний PID', p.pid)
    # Дождёмся выполнения процесса
    p.join()
    print('Завершение работы')

Пока выполняется процесс, съешьте ещё этих мягких французских булок да выпейте чаю ☕️ 
Главный PID 57336
Дочерний PID 61197
Привет Алиса
Завершение работы


При работе с обычными процессами программа будет ожидать завершения всех дочерних процессов, прежде чем сама сможет завершиться. 

Чтобы приложение дождалось завершения процесса-демона (работают в фоновом режиме и завершаются вместе с родительским процессом), нужно использовать p.join()

```sh
(base) aleksioprime@gymnmac knowledge % ps aux | grep 60262
aleksioprime     60441   0.0  0.0 435299840   1264 s004  R+   12:47PM   0:00.00 grep 60262
aleksioprime     60262   0.0  0.1 410320928   8480 s002  S+   12:47PM   0:00.05 /opt/anaconda3/bin/python mp_demo_1.py
(base) aleksioprime@gymnmac knowledge % ps aux | grep 60715
aleksioprime     60743   0.0  0.0 435298848    864 s004  R+   12:47PM   0:00.00 grep 60715
aleksioprime     60715   0.0  0.2 410752032  15520 s002  S+   12:47PM   0:00.04 /opt/anaconda3/bin/python -c from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=4, pipe_handle=6) --multiprocessing-fork
```

При запуске главного процесса интерпретатору передаётся название файла
При запуске дочернего процесса (методом spawn) интерпретатору передаётся код на выполнение `-c from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=4, pipe_handle=6)`, где выполняется функция `spawn_main` с аргументами `pipe_handle` (номер файл-дескриптора, через который будут читаться данные от родителя (через pipe)) и `tracker_fd` - (дескриптор для связи с resource_tracker, который отслеживает и чистит разделяемые ресурсы типа shared_memory, семафоров, некоторых shared-объектов (чтобы не оставались “утечки” системных ресурсов)). Флаг --multiprocessing-fork — внутренний маркер multiprocessing, указывающий, что процесс запущен как дочерний служебным механизмом multiprocessing (несмотря на название, применяется не только при fork).

При работе с процесами дочернему процессу можно передать объекты, которые могут быть сериализованы в массив-байт и десериализованы обратно. Для этого используется модуль `pickle`, который может работать в основном с примитивными данными.

## Очереди

Очереди позволяют обмениваться сообщениями сразу между группами процессов.

In [ ]:
import multiprocessing as mp

from multiprocessing import Process
from multiprocessing.queues import Queue
from time import sleep


def producer(queue: Queue):
    # Процесс-производитель, который будет писать сообщения в очередь
    while True:
        message = 'ping'
        queue.put(message)
        sleep(1)


def consumer(queue: Queue):
    # Процесс-потребитель, который будет читать сообщения из очереди и печатать их
    while message := queue.get():
        print(message)


if __name__ == '__main__':
    # Указание контекста spawn для запуска нового процесса интерпретатора Python
    ctx = mp.get_context('spawn')
    # Создаём очередь для обмена сообщениями между процессами
    queue = Queue(ctx=ctx)
    # Каждому процессу передаём ссылку на очередь
    producer_ = Process(target=producer, args=(queue,))
    consumer_ = Process(target=consumer, args=(queue,))
    # Один процесс пишет в очередь сообщение ping, а другой — вычитывает сообщение из очереди и печатает его в консоль.
    # Оба процесса работают бесконечно, поэтому мы не будем их останавливать, а просто запустим и дадим им поработать некоторое время.
    producer_.start()
    consumer_.start()
    producer_.join()
    consumer_.join()

ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping
ping


Process Process-11:
Process Process-10:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Users/aleksioprime/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/aleksioprime/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/aleksioprime/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/aleksioprime/.pyenv/versions/3.12.11/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/var/folders/ww/my_rn7cx4g3_6y5ybtlvj26r0000gn/T/ipykernel_57336/3944873599.py", line 16, in consumer
    while message := queue.get():
                     ^^^^^^^^^^^
  File "/var/folders/ww/my_rn7cx4g3_6y5ybtlvj26r0000gn/T/ipykernel_57336/3944873599.py", line 12, in producer
    sleep(1)
  

KeyboardInterrupt: 